In [ ]:
from google.colab import userdata
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
import os
client = QdrantClient(url=userdata.get("QDRANT_URL"), api_key=userdata.get("QDRANT_API_KEY"))

encoder = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
my_dataset = [
    {
        "title": "Classic Beef Bourguignon",
        "description": """A rich, wine-braised beef stew from Burgundy, France.
        Tender chunks of beef are slowly simmered with pearl onions, mushrooms,
        and bacon in a deep red wine sauce. The long, slow cooking process
        develops complex flavors and creates a luxurious, velvety texture.
        Perfect for cold winter evenings when you want something hearty and
        comforting. Traditionally served with crusty bread or creamy mashed
        potatoes to soak up the incredible sauce.""",
        "cuisine": "French",
        "difficulty": "Intermediate",
        "time": "3 hours"
    },
    {
        "title": "Spaghetti Carbonara",
        "description": """A classic Roman pasta dish made with eggs, Pecorino Romano
        cheese, pancetta, and freshly cracked black pepper. The hot pasta gently
        cooks the egg mixture, creating a silky, creamy sauce without using cream.
        Crispy pancetta adds a savory depth while the cheese provides a sharp,
        salty bite. Simple ingredients come together in minutes, making it an
        elegant yet comforting meal.""",
        "cuisine": "Italian",
        "difficulty": "Beginner",
        "time": "25 minutes"
    },
    {
        "title": "Chicken Tikka Masala",
        "description": """Tender pieces of marinated chicken are grilled until
        slightly charred, then simmered in a creamy tomato-based sauce infused
        with garam masala, cumin, coriander, and turmeric. The sauce is rich,
        slightly smoky, and perfectly spiced with a gentle heat. Often served
        with fluffy basmati rice or warm naan bread, it is a comforting and
        flavorful favorite around the world.""",
        "cuisine": "Indian",
        "difficulty": "Intermediate",
        "time": "1 hour"
    },
    {
        "title": "Sushi Platter",
        "description": """An assortment of fresh sushi including nigiri, maki rolls,
        and sashimi. Featuring vinegared rice paired with fresh salmon, tuna,
        avocado, and crisp cucumber. Precision and balance are key, with each
        bite delivering harmony between texture and flavor. Served with soy
        sauce, pickled ginger, and wasabi for a complete Japanese dining experience.""",
        "cuisine": "Japanese",
        "difficulty": "Advanced",
        "time": "1.5 hours"
    },
    {
        "title": "Mexican Chicken Tacos",
        "description": """Soft corn tortillas filled with juicy, spice-rubbed
        chicken grilled to perfection. Topped with fresh pico de gallo, creamy
        avocado slices, cilantro, and a squeeze of lime. The combination of
        smoky meat and bright toppings creates a vibrant, street-style taco
        experience that is both satisfying and refreshing.""",
        "cuisine": "Mexican",
        "difficulty": "Beginner",
        "time": "35 minutes"
    },
    {
        "title": "Thai Green Curry",
        "description": """A fragrant and spicy coconut-based curry made with green
        curry paste, tender chicken or vegetables, Thai basil, and bamboo shoots.
        The creamy coconut milk balances the heat of the chilies, while fish
        sauce and lime add brightness and depth. Served over jasmine rice for a
        comforting and aromatic meal.""",
        "cuisine": "Thai",
        "difficulty": "Intermediate",
        "time": "45 minutes"
    },
    {
        "title": "Greek Moussaka",
        "description": """A layered casserole of roasted eggplant, spiced ground
        lamb, and creamy béchamel sauce baked until golden and bubbling. The
        cinnamon and oregano in the meat sauce add warmth and complexity, while
        the velvety top layer provides richness. A hearty Mediterranean dish
        perfect for family gatherings.""",
        "cuisine": "Greek",
        "difficulty": "Advanced",
        "time": "2 hours"
    },
    {
        "title": "American Cheeseburger",
        "description": """A juicy beef patty grilled to perfection and topped with
        melted cheddar cheese, crisp lettuce, ripe tomato, and tangy pickles.
        Served in a toasted brioche bun with ketchup and mustard. The balance of
        savory meat, creamy cheese, and fresh toppings makes this a timeless
        comfort food classic.""",
        "cuisine": "American",
        "difficulty": "Beginner",
        "time": "20 minutes"
    },
    {
        "title": "Spanish Paella",
        "description": """A vibrant rice dish cooked in a wide, shallow pan with
        saffron, seafood, chicken, and chorizo. The rice absorbs a rich broth
        infused with paprika and garlic, forming a crispy, caramelized layer
        called socarrat at the bottom. Colorful and aromatic, it captures the
        spirit of Spanish coastal cuisine.""",
        "cuisine": "Spanish",
        "difficulty": "Advanced",
        "time": "1.5 hours"
    },
    {
        "title": "Moroccan Lamb Tagine",
        "description": """Slow-cooked lamb infused with warm spices like cinnamon,
        cumin, and ginger, combined with dried apricots and almonds for a
        sweet-savory contrast. Cooked in a traditional clay pot, the dish
        becomes tender and deeply flavorful. Served with fluffy couscous to
        absorb the fragrant sauce.""",
        "cuisine": "Moroccan",
        "difficulty": "Intermediate",
        "time": "2.5 hours"
    },
    {
        "title": "Chinese Kung Pao Chicken",
        "description": """A bold and spicy stir-fry featuring diced chicken, dried
        red chilies, roasted peanuts, and crisp vegetables in a savory-sweet
        sauce. The combination of heat, crunch, and umami creates a dynamic
        flavor profile. Traditionally served with steamed rice for balance.""",
        "cuisine": "Chinese",
        "difficulty": "Intermediate",
        "time": "40 minutes"
    },
    {
        "title": "Vegetable Ratatouille",
        "description": """A rustic French vegetable stew made with zucchini,
        eggplant, tomatoes, and bell peppers simmered in olive oil with garlic
        and fresh herbs. The vegetables become tender while maintaining their
        individual character, creating a vibrant and healthy dish. Delicious
        served warm or at room temperature with crusty bread.""",
        "cuisine": "French",
        "difficulty": "Beginner",
        "time": "1 hour"
    }
]

In [ ]:
def fixed_size_chunks(text, chunk_size=100, overlap=20):
    """Split text into fixed-size chunks with overlap"""
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size - overlap):
        chunk_words = words[i:i + chunk_size]
        if chunk_words:  # Only add non-empty chunks
            chunks.append(' '.join(chunk_words))

    return chunks

def sentence_chunks(text, max_sentences=3):
    """Group sentences into chunks"""
    import re
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk_sentences = sentences[i:i + max_sentences]
        if chunk_sentences:
            chunks.append('. '.join(chunk_sentences) + '.')

    return chunks

def paragraph_chunks(text):
    """Split by paragraphs or double line breaks"""
    chunks = [chunk.strip() for chunk in text.split('\n\n') if chunk.strip()]
    return chunks if chunks else [text]  # Fallback to full text

In [ ]:
collection_name = "day1_semantic_search"

if client.collection_exists(collection_name=collection_name):
    client.delete_collection(collection_name=collection_name)

# Create a collection with three named vectors
client.create_collection(
    collection_name=collection_name,
    vectors_config={
        "fixed": models.VectorParams(size=384, distance=models.Distance.COSINE),
        "sentence": models.VectorParams(size=384, distance=models.Distance.COSINE),
        "paragraph": models.VectorParams(size=384, distance=models.Distance.COSINE),
    },
)

# Index fields for filtering (more on this on day 2)
client.create_payload_index(
    collection_name=collection_name,
    field_name="chunk_strategy",
    field_schema=models.PayloadSchemaType.KEYWORD,
)

# Process and upload data
points = []
point_id = 0

for item in my_dataset:
    description = item["description"]

    # Process with each chunking strategy
    strategies = {
        "fixed": fixed_size_chunks(description),
        "sentence": sentence_chunks(description),
        "paragraph": paragraph_chunks(description),
    }

    for strategy_name, chunks in strategies.items():
        for chunk_idx, chunk in enumerate(chunks):
            # Create vectors for this chunk
            vectors = {strategy_name: encoder.encode(chunk).tolist()}

            points.append(
                models.PointStruct(
                    id=point_id,
                    vector=vectors,
                    payload={
                        **item,  # Include all original metadata
                        "chunk": chunk,
                        "chunk_strategy": strategy_name,
                        "chunk_index": chunk_idx,
                    },
                )
            )
            point_id += 1

client.upload_points(collection_name=collection_name, points=points)
print(f"Uploaded {len(points)} chunks across three strategies")

Uploaded 39 chunks across three strategies


In [ ]:
def compare_search_results(query):
    """Compare search results across all chunking strategies"""
    print(f"Query: '{query}'\n")

    for strategy in ["fixed", "sentence", "paragraph"]:
        results = client.query_points(
            collection_name=collection_name,
            query=encoder.encode(query).tolist(),
            using=strategy,
            limit=3,
        )

        print(f"--- {strategy.upper()} CHUNKING ---")
        for i, point in enumerate(results.points, 1):
            print(f"{i}. {point.payload['title']}")
            print(f"   Score: {point.score:.3f}")
            print(f"   Chunk: {point.payload['chunk'][:80]}...")
        print()


# Test with domain-specific queries
test_queries = [
    "comfort food for winter",  # Adapt these to your domain
    "quick and easy weeknight dinner",
    "elegant dish for special occasions",
]

for query in test_queries:
    compare_search_results(query)

Query: 'comfort food for winter'

--- FIXED CHUNKING ---
1. Vegetable Ratatouille
   Score: 0.377
   Chunk: A rustic French vegetable stew made with zucchini, eggplant, tomatoes, and bell ...
2. American Cheeseburger
   Score: 0.371
   Chunk: A juicy beef patty grilled to perfection and topped with melted cheddar cheese, ...
3. Chinese Kung Pao Chicken
   Score: 0.364
   Chunk: A bold and spicy stir-fry featuring diced chicken, dried red chilies, roasted pe...

--- SENTENCE CHUNKING ---
1. Classic Beef Bourguignon
   Score: 0.557
   Chunk: Perfect for cold winter evenings when you want something hearty and 
        com...
2. Sushi Platter
   Score: 0.393
   Chunk: Served with soy 
        sauce, pickled ginger, and wasabi for a complete Japane...
3. Vegetable Ratatouille
   Score: 0.377
   Chunk: A rustic French vegetable stew made with zucchini, 
        eggplant, tomatoes, ...

--- PARAGRAPH CHUNKING ---
1. Vegetable Ratatouille
   Score: 0.377
   Chunk: A rustic French vegetable ste

In [ ]:
def analyze_chunking_effectiveness():
    """Analyze which chunking strategy works best for your domain"""

    print("CHUNKING STRATEGY ANALYSIS")
    print("=" * 40)

    # Get chunk statistics for each strategy
    for strategy in ["fixed", "sentence", "paragraph"]:
        # Count chunks per strategy
        results = client.scroll(
            collection_name=collection_name,
            scroll_filter=models.Filter(
                must=[
                    models.FieldCondition(
                        key="chunk_strategy", match=models.MatchValue(value=strategy)
                    )
                ]
            ),
            limit=100,
        )

        chunks = results[0]
        chunk_sizes = [len(chunk.payload["chunk"]) for chunk in chunks]

        print(f"\n{strategy.upper()} STRATEGY:")
        print(f"  Total chunks: {len(chunks)}")
        print(f"  Avg chunk size: {sum(chunk_sizes)/len(chunk_sizes):.0f} chars")
        print(f"  Size range: {min(chunk_sizes)}-{max(chunk_sizes)} chars")


analyze_chunking_effectiveness()

CHUNKING STRATEGY ANALYSIS

FIXED STRATEGY:
  Total chunks: 12
  Avg chunk size: 323 chars
  Size range: 261-437 chars

SENTENCE STRATEGY:
  Total chunks: 15
  Avg chunk size: 286 chars
  Size range: 95-413 chars

PARAGRAPH STRATEGY:
  Total chunks: 12
  Avg chunk size: 361 chars
  Size range: 288-491 chars
